# Read a forecast model run collection with Rolodex

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ocean-icechunks/2026-ioos-hackathon/blob/main/notebooks/gomofs-fmrc-rolodex.ipynb)

This notebook reads the store built by [`gomofs-fmrc-icechunk.ipynb`](gomofs-fmrc-icechunk.ipynb): three
GoMOFS forecast cycles, with `time` (the cycle time) and `step` (the lead time, 0 to 72 hours). **Run
that notebook first, on the same machine and in the same folder**, so the store `gomofs-fmrc-icechunk`
is here.

A collection like this has the same valid time in several runs. [Rolodex](https://github.com/dcherian/rolodex)
adds an xarray index that lets you ask for a view of the collection: a **best estimate**, one continuous
series that uses the newest run for every hour.

In [ ]:
%pip install -q "icechunk>=2.2" "xarray>=2026.7" "zarr>=3.4" obstore obspec_utils "dask[array]" matplotlib "git+https://github.com/dcherian/rolodex@527bd420b21e9efd88cb1601e58276d24312d040"

## 1. Open the store

The store holds only pointers into NOAA's NetCDF files, so the reader has to say that it may fetch
chunks from NOAA's bucket.

In [ ]:
import warnings

import icechunk
import xarray as xr

warnings.filterwarnings("ignore", category=UserWarning)

path = "gomofs-fmrc-icechunk"
bucket = "s3://noaa-nos-ofs-pds/"
creds = icechunk.containers_credentials({bucket: icechunk.s3_credentials(anonymous=True)})
repo = icechunk.Repository.open(icechunk.local_filesystem_storage(path), authorize_virtual_chunk_access=creds)
ds = xr.open_zarr(repo.readonly_session("main").store, consolidated=False, chunks={})
ds[["time", "step"]]

## 2. Add the valid time

The store has two time axes, `time` (the cycle) and `step` (the lead). The time a value is *valid* is
`time + step`, which is never stored. Rolodex builds it lazily, as a 2-D variable, and indexes all
three together.

In [ ]:
import rolodex.forecast
from rolodex.forecast import BestEstimate, ForecastIndex

ds.coords["valid_time"] = rolodex.forecast.create_lazy_valid_time_variable(
    reference_time=ds.time, period=ds.step
)
fmrc = ds.drop_indexes(["time", "step"]).set_xindex(["time", "step", "valid_time"], ForecastIndex)
fmrc["valid_time"]

## 3. The best estimate

`BestEstimate` takes, for each valid time, the data from the newest run that covers it. With
`offset=6` we start each run at step 6, which is 18 hours in, so the first hours of every run, while the
model settles after its start, are skipped. The result has one dimension, `valid_time`, and `time` and
`step` tell you which run and lead each value came from.

In [ ]:
best = fmrc.sel(valid_time=BestEstimate(offset=6))
print(f"{best.sizes['valid_time']} valid times, {best.valid_time.values[0]} to {best.valid_time.values[-1]}")
best[["time", "step"]].to_dataframe().iloc[::4]

## 4. Pick a place

The model grid is curvilinear, so we find the grid cell nearest a point with its 2-D longitude and
latitude. We use the NERACOOS buoy B01, off Cape Elizabeth, Maine.

In [ ]:
import numpy as np

lon, lat = -70.427, 43.180
dist = np.hypot((ds.lon_rho - lon) * np.cos(np.deg2rad(lat)), ds.lat_rho - lat)
j, i = np.unravel_index(int(dist.argmin()), dist.shape)
print(f"nearest cell: eta_rho={j}, xi_rho={i}, {float(ds.lon_rho[j, i]):.3f}, {float(ds.lat_rho[j, i]):.3f}")

## 5. Best estimate against the individual runs

We read the sea surface height `zeta` at that cell, for the best estimate and for each run on its own.
Each `zeta` chunk covers the whole model domain, so this reads a few MB per time step.

In [ ]:
%%time
zeta_best = best["zeta"].isel(eta_rho=j, xi_rho=i).compute()
zeta_runs = [ds["zeta"].isel(time=k, eta_rho=j, xi_rho=i).compute() for k in range(ds.sizes["time"])]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 3.5))
for k, run in enumerate(zeta_runs):
    ax.plot(ds.time.values[k] + run.step.values, run, lw=1, alpha=0.6, label=f"run {ds.time.values[k].astype('datetime64[h]')}")
ax.plot(zeta_best.valid_time, zeta_best, "k", lw=2.5, label="best estimate")
ax.set(ylabel="sea surface height (m)", title="GoMOFS zeta near buoy B01")
ax.grid(alpha=0.3)
ax.legend()
fig.autofmt_xdate()

## What you have

One store, every run. The thin lines are the individual forecasts; the thick line takes, for each hour, the
newest run that covers it. When the next cycle is appended along `time`, the same lines of code give
a best estimate that extends to include it, with no change to anything already in the store.